In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = pd.read_csv("traffic_analysis_grouped_vio.csv")

df

/var/folders/kt/_bzbz3vd5qbcfjnt5vssmqxh0000gn/T/ipykernel_3531/163268045.py:10: DtypeWarning: Columns (12,13,18,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("traffic_analysis_grouped_vio.csv")


,raw_row_number,date,time,location,lat,lng,geocode_source,beat,district,subject_age,...,trust_in_local_government_rate,economic_diversity_index,hardship_index,median_household_income,poverty_rate,foreign_born,limited_english_proficiency,demographics,population,grouped_vio
0,4,2012-01-01,01:00:00,108 LAMON AVE N,41.882072,-87.747829,SU,NaN,NaN,35.0,...,39.844682,0.775025,85.469151,51826.851402,23.399157,10.704508,8.804414,5.533701,97671.110800,Moving Violations
1,9,2012-01-01,01:00:00,430 STATE ST N,41.890322,-87.628217,GM,NaN,NaN,26.0,...,39.185753,0.354411,3.402425,125101.326886,9.622478,18.181435,4.297681,66.267215,99292.486072,Pedestrian/Non-Driver
2,11,2012-01-01,01:00:00,4225 ARMITAGE AVE W,41.916806,-87.732301,GM,NaN,NaN,26.0,...,53.921078,0.798612,77.967935,70459.881153,13.674981,34.878304,36.024369,9.575501,22644.717836,License Issues
3,18,2012-01-01,03:00:00,1011 GARFIELD BLVD W,41.793582,-87.650526,GM,NaN,NaN,22.0,...,27.250549,0.728528,93.089143,32084.861861,33.435182,5.675052,5.465415,1.005840,23160.939521,License Issues
4,20,2012-01-01,02:00:00,13455 MACKINAW AVE S,41.650292,-87.542070,GM,NaN,NaN,21.0,...,28.253638,0.802099,68.392927,61745.599215,9.820789,14.676821,11.130220,35.767045,8767.938562,License Issues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1877393,2108093,2020-05-16,23:37:00,7XX S HOMAN AVE,41.880526,-87.710980,SU,1134.0,11.0,23.0,...,22.923965,0.732096,92.150919,39861.333805,37.256195,2.921534,1.897934,7.756904,20739.825817,Moving Violations
1877394,2108095,2020-05-16,23:40:00,60XX S STATE ST,41.881017,-87.627817,SU,311.0,3.0,34.0,...,48.260009,0.303370,6.880908,119221.584097,11.145882,27.777929,4.824201,49.764161,43756.849875,Equipment
1877395,2108096,2020-05-16,23:40:00,26XX S STATE ST,41.881212,-87.628180,SU,133.0,1.0,31.0,...,48.260009,0.303370,6.880908,119221.584097,11.145882,27.777929,4.824201,49.764161,43756.849875,Speeding
1877396,2108097,2020-05-16,23:47:00,44XX N SAWYER AVE,41.898274,-87.707719,SU,1724.0,17.0,44.0,...,32.277557,0.781104,82.767377,58409.067093,26.772810,22.184584,21.887965,12.249829,55766.955333,Equipment


In [2]:
# create binary outcome
df["citation_binary"] = df["citation_issued"].fillna(False).astype(int)

# keep only needed columns
model_df = df[
    ["citation_binary", "subject_sex", "community_area_id", "grouped_vio"]
].copy()

# clean subject_sex
model_df["subject_sex"] = model_df["subject_sex"].str.strip().str.lower()

# split predictors and response
X = model_df.drop(columns="citation_binary")
y = model_df["citation_binary"]

# define column types
categorical_features = ["subject_sex", "community_area_id", "grouped_vio"]

# preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categorical_features)
])

# full model pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=X["community_area_id"]))

X_train= X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# fit model
model.fit(X_train, y_train)

# predicted probabilities
y_pred_prob = model.predict_proba(X_test)[:, 1]

# evaluate
auc = roc_auc_score(y_test, y_pred_prob)
print("ROC AUC:", auc)

ROC AUC: 0.5764734946406671
